# 额外的周末练习 —— 第 2 周

## 练习目标（理念）

用你在 **第 2 周**学到的能力，把第 1 周的「技术问答器」做成更完整的原型：

- **Gradio UI**：可交互的聊天界面
- **流式（streaming）**：一边生成一边显示（本实现用「按词 yield」模拟打字机）
- **system prompt**：注入技术导师人设
- **多模型切换**：下拉框在 `gpt-4.1-mini` 与 `gpt-5-nano` 之间切换
- **奖励分**：演示 **工具调用（tool calling）**——按 URL 抓网页再讲解/摘要
- **更大胆**：音频输入 / 音频回复（可选，本笔记本未实现）

## 和本课概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions | `openai.chat.completions.create(...)` |
| system / user / history | 拼完整 `messages` |
| Function / Tool Calling | `tools` + `finish_reason == "tool_calls"` |
| Gradio Blocks + ChatInterface | 模型下拉 + 聊天窗口 |

## 怎么跑

1. 准备好 `.env` 里的 `OPENAI_API_KEY`
2. 同目录需有 `scraper.fetch_website_contents`（或按你的实现调整）
3. 从上到下运行单元格，最后 `demo.launch()` 打开界面


In [ ]:
# ========== 导入：环境、OpenAI、Gradio、抓网页工具 ==========

# os：读环境变量（Environment Variables）
import os
# json：解析 tool call 的 arguments（常是 JSON 字符串）
import json
# load_dotenv：把 .env 密钥读进进程环境
from dotenv import load_dotenv
# OpenAI 客户端：调用 Chat Completions + tools
from openai import OpenAI
# gradio：搭聊天 Web UI
import gradio as gr
# 抓取网页正文（给 LLM 当 tool 用）
from scraper import fetch_website_contents


In [ ]:
# ========== 初始化：加载密钥、建客户端、准备模型下拉选项 ==========

# override=True：.env 覆盖已有同名环境变量
load_dotenv(override=True)
# 读取 OpenAI API Key
openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    # 只打印前缀，确认密钥存在
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

# 默认从环境变量读密钥创建客户端
openai = OpenAI()
# 下拉显示名 → API 真正的 model id（本练习只用 OpenAI）
MODELS = {"gpt-4.1-mini": "gpt-4.1-mini", "gpt-5-nano (faster/cheaper)": "gpt-5-nano"}
# 用单元素 list 存「当前选中的下拉标签」，便于在闭包里就地修改
current_model = ["gpt-4.1-mini"]  # default (dropdown label); updated by dropdown, mapped to id in chat()


In [ ]:
# ========== system prompt：技术导师人设（发给模型的指令，保留英文） ==========

# 说明能答什么、如何用 markdown、何时调用 fetch_url_content 工具
system_message = """You are a helpful technical tutor. You answer questions about Python, software engineering, data science, and LLMs in a clear, concise way.
Give accurate explanations. If you don't know something, say so. Use markdown for structure (headers, lists, code snippets) when helpful.
When the user shares a URL, you may use the fetch_url_content tool to get the page content and then explain or summarize it. Use the tool when it would help answer their question."""


In [ ]:
# ========== 工具实现 + JSON Schema：告诉模型何时/如何抓 URL ==========

def fetch_url_content(url: str) -> str:
    """Fetch and return the text content of a webpage. Use when the user asks about a URL."""
    try:
        # 真正干活：调用同目录 scraper
        return fetch_website_contents(url)
    except Exception as e:
        # 失败时把错误信息当字符串返回给模型（文案保留英文）
        return f"Error fetching URL: {e}"

# OpenAI tools 要求的 function 描述（name / description / parameters）
fetch_url_function = {
    "name": "fetch_url_content",
    "description": "Fetch the text content of a webpage at the given URL. Use when the user asks to explain, summarize, or read a web page.",
    "parameters": {
        "type": "object",
        "properties": {
            "url": {"type": "string", "description": "Full URL of the page (e.g. https://example.com)"},
        },
        "required": ["url"],
        # 禁止额外未知字段，减少胡编参数
        "additionalProperties": False,
    },
}
# 挂到 chat.completions.create(..., tools=...)
tools = [{"type": "function", "function": fetch_url_function}]


In [ ]:
# ========== handle_tool_calls：执行模型请求的函数并拼回 tool 消息 ==========

def handle_tool_calls(message):
    responses = []
    # 一次回复里可能有多个 tool_calls
    for tool_call in message.tool_calls:
        if tool_call.function.name == "fetch_url_content":
            # arguments 是 JSON 字符串 → dict
            args = json.loads(tool_call.function.arguments)
            url = args.get("url", "")
            # 本地真正抓网页
            content = fetch_url_content(url)
            # role=tool：把结果连同 tool_call_id 交回模型
            responses.append({
                "role": "tool",
                "content": content,
                "tool_call_id": tool_call.id,
            })
    return responses


In [ ]:
# ========== chat：tool-call 循环 + 按词 yield 模拟流式 ==========

def chat(message, history):
    # 下拉标签 → 真实 model id；找不到就原样用
    model = MODELS.get(current_model[0], current_model[0])  # map dropdown label to API model id
    # Gradio messages history → 只要 role/content
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    # system + 历史 + 当前 user
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    # 第一次请求（带 tools，模型可选择调用）
    response = openai.chat.completions.create(model=model, messages=messages, tools=tools)

    # 循环：模型要工具 → 本地执行 → 再问模型，直到不再 tool_calls
    while response.choices[0].finish_reason == "tool_calls":
        msg = response.choices[0].message
        tool_responses = handle_tool_calls(msg)
        # 先追加助手的 tool_calls 消息
        messages.append(msg)
        # 再追加每条 tool 结果
        messages.extend(tool_responses)
        response = openai.chat.completions.create(model=model, messages=messages, tools=tools)

    # 最终文本：按空格切词逐步 yield（打字机效果；真正 token 流式可另改）
    final_content = response.choices[0].message.content or ""
    if not final_content:
        yield "(No text reply)"
        return
    result = ""
    for word in final_content.split():
        result += word + " "
        yield result


In [ ]:
# ========== Gradio Blocks：模型下拉 + ChatInterface ==========

def set_model(m):
    # 下拉变更时写入 current_model[0]，供 chat() 读取
    current_model[0] = m
    return m

with gr.Blocks(title="Technical Q&A Tutor", theme=gr.themes.Soft()) as demo:
    # 界面说明（字符串保留英文，避免改可运行 UI 文案行为）
    gr.Markdown("## Week 2 – Technical Q&A Tutor\nAsk anything about Python, software engineering, data science, or LLMs. Paste a URL and ask to explain or summarize it—the assistant can fetch the page.")
    model_selector = gr.Dropdown(
        choices=list(MODELS.keys()),
        value="gpt-4.1-mini",
        label="Model",
        info="Switch between faster/cheaper (gpt-5-nano) and stronger (gpt-4.1-mini)",
    )
    # 选中变化 → 更新 current_model
    model_selector.change(fn=set_model, inputs=model_selector)
    gr.ChatInterface(
        fn=chat,
        type="messages",
        examples=[
            "Explain what a Python generator is and when to use it.",
            "What does this code do? x = [n**2 for n in range(10)]",
            "Summarize this page: https://www.python.org/about/",
        ],
    )

# 启动本地 Gradio 服务
demo.launch()
